In [ ]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final')

In [ ]:
import re

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

male_forms = ["потерпевший", "потерпевшему", "потерпевшего", "потерпевшем"]
male_pattern = r"\b(" + "|".join(male_forms) + r")\b"

train_data_has_man_victim = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    has_man = row["has_man_victim"]
    if pd.isnull(has_man):
        continue

    if has_man == 1:
        match = re.search(male_pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_has_man_victim.append((text, {"entities": [(start, end, "HAS_MAN_VICTIM")]}))
            print(text[start:end])
        else:
            print(f"Не найдена форма 'потерпевший' при has_man_victim = 1 в id={row['id']}")
    else:
        train_data_has_man_victim.append((text, {"entities": []}))
        pass

print(f"TRAIN_DATA_HAS_MAN_VICTIM готово: {len(train_data_has_man_victim)} примеров")

потерпевшему
Потерпевший
Потерпевший
потерпевшему
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшему
потерпевшего
потерпевшего
Потерпевший
потерпевшего
потерпевшему
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
Потерпевший
потерпевшему
потерпевшего
потерпевшего
потерпевшему
потерпевшего
потерпевшего
Потерпевший
потерпевшего
потерпевшего
потерпевшего
потерпевшему
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшему
Потерпевший
потерпевшего
потерпевший
Потерпевший
Потерпевший
потерпевшего
Потерпевший
потерпевшего
потерпевшему
потерпевшего
потерпевшему
потерпевшего
потерпевшему
потерпевшему
потерпевшего
потерпевшего
потерпевшему
Потерпевший
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевший
Потерпевший
потерпевшего
потерпевший
потерпевшего
потерпевшему
потерпевшего
потерпевший
потерпевшего
потерпевшему
потерпевшего
потерпевшему
потерпевшего
Потерпевший
Потерпевший
TRAIN_DATA_HAS_MAN_VICTIM готово: 100 прим

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("HAS_MAN_VICTIM")

examples = []
for text, annot in train_data_has_man_victim:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(10):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_has_man_victim_model")
print("Модель сохранена в 'ner_has_man_victim_model'")

Epoch 1, Losses: {'ner': 263073.214650198}
Epoch 2, Losses: {'ner': 153.95392508809235}
Epoch 3, Losses: {'ner': 389.79263348013194}
Epoch 4, Losses: {'ner': 2320.858240845062}
Epoch 5, Losses: {'ner': 173.67165346267447}
Epoch 6, Losses: {'ner': 132.9116383343076}
Epoch 7, Losses: {'ner': 115.32310578989326}
Epoch 8, Losses: {'ner': 103.86001383736097}
Epoch 9, Losses: {'ner': 105.61557922739328}
Epoch 10, Losses: {'ner': 85.57346620589686}
Модель сохранена в 'ner_has_man_victim_model'


In [14]:
for i in range(11, 16):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

Epoch 12, Losses: {'ner': 89.68765568170132}
Epoch 13, Losses: {'ner': 86.59788470531528}
Epoch 14, Losses: {'ner': 77.59394924203951}
Epoch 15, Losses: {'ner': 64.59697930800176}
Epoch 16, Losses: {'ner': 58.97334244739664}


In [16]:
for i in range(16, 18):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

Epoch 17, Losses: {'ner': 90.44936343818321}
Epoch 18, Losses: {'ner': 93.83164282489723}


In [15]:
from sklearn.metrics import accuracy_score, f1_score

nlp = spacy.load("ner_has_man_victim_model")

y_true = []
y_pred = []

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

for _, row in df_marking.iterrows():
    true_val = row["has_man_victim"]
    if pd.isnull(true_val):
        continue

    text = get_full_text(row)
    doc = nlp(text.lower())

    predicted_val = 0
    for ent in doc.ents:
        if ent.label_ == "HAS_MAN_VICTIM":
            print(ent.text)
            predicted_val = 1
            break

    y_true.append(int(true_val))
    y_pred.append(predicted_val)

# Метрики
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.2%}")
print(f"F1-score: {f1:.2%}")

потерпевшему
потерпевший
потерпевший
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевший
потерпевшего
потерпевшего
потерпевший
потерпевшего
потерпевшему
потерпевший
потерпевшего
потерпевший
потерпевший
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевшего
потерпевший
потерпевший
потерпевший
потерпевший
потерпевшего
потерпевшему
потерпевший
потерпевшему
потерпевший
потерпевшему
потерпевший
потерпевший
потерпевший
потерпевший
потерпевшего
потерпевший
потерпевший
потерпевшему
потерпевшего
потерпевшего
потерпевший
потерпевший
потерпевшему
потерпевшего
потерпевший
потерпевший
потерпевший
потерпевший
Accuracy: 59.00%
F1-score: 61.75%


In [18]:
sum = 0
for idx, (true_val, pred_val) in enumerate(zip(y_true, y_pred)):
    if true_val != pred_val:
        print(f"Ошибка в индексе {idx}: true={true_val}, pred={pred_val}")
        sum += 1
print(sum)

Ошибка в индексе 4: true=1, pred=0
Ошибка в индексе 8: true=0, pred=1
Ошибка в индексе 9: true=1, pred=0
Ошибка в индексе 11: true=1, pred=0
Ошибка в индексе 12: true=1, pred=0
Ошибка в индексе 18: true=1, pred=0
Ошибка в индексе 20: true=1, pred=0
Ошибка в индексе 25: true=0, pred=1
Ошибка в индексе 27: true=1, pred=0
Ошибка в индексе 29: true=1, pred=0
Ошибка в индексе 30: true=1, pred=0
Ошибка в индексе 32: true=1, pred=0
Ошибка в индексе 33: true=1, pred=0
Ошибка в индексе 36: true=1, pred=0
Ошибка в индексе 37: true=1, pred=0
Ошибка в индексе 40: true=1, pred=0
Ошибка в индексе 42: true=1, pred=0
Ошибка в индексе 43: true=1, pred=0
Ошибка в индексе 45: true=1, pred=0
Ошибка в индексе 52: true=0, pred=1
Ошибка в индексе 53: true=1, pred=0
Ошибка в индексе 57: true=0, pred=1
Ошибка в индексе 58: true=1, pred=0
Ошибка в индексе 60: true=0, pred=1
Ошибка в индексе 61: true=1, pred=0
Ошибка в индексе 62: true=1, pred=0
Ошибка в индексе 64: true=1, pred=0
Ошибка в индексе 66: true=1, pr